# Tapi · Exploración inicial con SQL

Ejecutá las celdas en orden. Usamos DuckDB para consultar los CSV con SQL dentro de Colab.

Los archivos se descargan automáticamente desde DaianaSep/tapi_Challenge, rama main. Como el repositorio es privado, configurá una vez el secreto GITHUB_TOKEN en el panel de secretos (llave) de Colab y habilitá el acceso de este notebook. Usá un token de GitHub con acceso de lectura al contenido de este repositorio. La descarga requiere que el token siga vigente.

Los tres CSV están en la raíz del repositorio. Cada ejecución descarga una copia actual de main y carga las tablas cuentas, consultas y pagos. Fuera de Colab, el token se puede suministrar mediante la variable de entorno GITHUB_TOKEN.


In [ ]:
%pip -q install duckdb==1.4.3

In [ ]:
from pathlib import Path
import os
import requests
import duckdb

REPOSITORIO = "DaianaSep/tapi_Challenge"
RAMA = "main"
nombres = {
    "cuentas": "B_agenda_cuentas.csv",
    "consultas": "B_agenda_consultas.csv",
    "pagos": "B_agenda_pagos.csv",
}

try:
    from google.colab import userdata
except ImportError:
    token = os.environ.get("GITHUB_TOKEN", "")
else:
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Agregá GITHUB_TOKEN en Secretos de Colab (icono de llave) "
            "y habilitá el acceso de este notebook."
        ) from None

if not token:
    raise RuntimeError("Falta GITHUB_TOKEN para leer el repositorio privado.")

carpeta = Path("data")
carpeta.mkdir(exist_ok=True)
descargas = {}
with requests.Session() as sesion:
    sesion.headers.update({
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github.raw+json",
        "X-GitHub-Api-Version": "2022-11-28",
    })
    for archivo in nombres.values():
        url = f"https://api.github.com/repos/{REPOSITORIO}/contents/{archivo}"
        respuesta = sesion.get(url, params={"ref": RAMA}, timeout=60)
        if respuesta.status_code != 200:
            raise RuntimeError(
                f"No se pudo descargar {archivo} (HTTP {respuesta.status_code}). "
                "Revisá la vigencia y el permiso de lectura del token, "
                "el nombre del repositorio y la rama."
            )
        descargas[archivo] = respuesta.content

del token

# Cargamos las tres tablas en una conexión nueva antes de reemplazar la anterior.
nueva_conexion = duckdb.connect()
try:
    for tabla, archivo in nombres.items():
        destino = carpeta / archivo
        temporal = destino.with_suffix(".csv.tmp")
        temporal.write_bytes(descargas[archivo])
        temporal.replace(destino)
        ruta = str(destino).replace("'", "''")
        nueva_conexion.execute(
            f"CREATE TABLE {tabla} AS "
            f"SELECT * FROM read_csv('{ruta}', header=true, all_varchar=true)"
        )
except Exception:
    nueva_conexion.close()
    raise

if "conexion" in globals():
    conexion.close()
conexion = nueva_conexion

def consultar(sql):
    return conexion.execute(sql).df()

print("CSV descargados desde GitHub. Tablas cuentas, consultas y pagos listas.")


Cargamos inicialmente todas las columnas como texto para conservar IDs y timestamps originales. Los campos vacíos se leen como NULL. Más adelante convertiremos fechas y montos para los cálculos que lo requieran.

## Muestra de cuentas

Cada fila representa una adhesión a la Agenda. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM cuentas
LIMIT 10
""")

## Muestra de consultas

Cada fila representa un intento de consulta. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM consultas
LIMIT 10
""")

## Muestra de pagos

Cada fila representa un pago realizado. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM pagos
LIMIT 10
""")

## Cantidad de registros

COUNT(*) cuenta filas. UNION ALL apila los resultados de las tres tablas.

In [ ]:
consultar("""
SELECT 'cuentas' AS tabla, COUNT(*) AS filas FROM cuentas
UNION ALL
SELECT 'consultas' AS tabla, COUNT(*) AS filas FROM consultas
UNION ALL
SELECT 'pagos' AS tabla, COUNT(*) AS filas FROM pagos
""")

## Identificadores de cuentas

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(account_id) AS filas_con_id,
    COUNT(DISTINCT account_id) AS cuentas_distintas
FROM cuentas
""")

## Identificadores de consultas

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(consulta_id) AS filas_con_id,
    COUNT(DISTINCT consulta_id) AS consultas_distintas
FROM consultas
""")

## Identificadores de pagos

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(pago_id) AS filas_con_id,
    COUNT(DISTINCT pago_id) AS pagos_distintos
FROM pagos
""")

## Convenios y proveedores

Una fila por combinación de convenio y proveedor. Permite identificar convenios compartidos y comparar el volumen de consultas.

In [ ]:
consultar("""
SELECT
    convenio,
    proveedor,
    COUNT(*) AS cantidad_consultas
FROM consultas
GROUP BY convenio, proveedor
ORDER BY convenio, proveedor
""")

## Tus observaciones

Anotá qué encontraste en cada consulta y qué querés investigar después. Revisaremos origen, resultado, fechas y campos vacíos en el próximo paso.